## Using `pyesgf`

A simpler, less streamlined, hopefully more controllable way to download cmip files: hopefully filling the gap between the janky `wget` script downloading and individual running, and the dodginess of `esgpull`.

`pysegf` has many more contributors, forks, and stars than `esgpull` (as of April 2025). I'll take this as a good sign.

In [ ]:
### imports
# general
import pandas as pd
from tqdm.auto import tqdm
import os
import re
# processing
import xarray as xa
# pyesgf
from pyesgf.search import SearchConnection

os.environ["ESGF_PYCLIENT_NO_FACETS_STAR_WARNING"] = "on"

## Log-on

Unnecessary for my datasets: only required for restricted data.

In [ ]:
# from pyesgf.logon import LogonManager

# OPENID_USERNAME = "Orlando.Timmerman"
# SERVER = "ceda.ac.uk"
# OPENID = f'https://esgf.{SERVER}/openid/esgf-idp/{OPENID_USERNAME}'
# password = "CEDAtrees2022."

# lm = LogonManager()
# lm.logon_with_openid(OPENID, password, bootstrap=True)

## Search

In [ ]:
from pyesgf.search import SearchConnection
new_conn = SearchConnection('https://esgf-node.llnl.gov/esg-search', distrib=True)
new_ctx = new_conn.new_context(
    project='CMIP6',
    experiment_id='historical,ssp126,ssp245,ssp370,ssp585',
    variable='arag,tos,rsdo,so,no3,po4',
    frequency='mon',
    )

print('Number of hits:', new_ctx.hit_count)
ctx_result = new_ctx.search()
individual_result = ctx_result[0]
individual_result_files = individual_result.file_context().search()
for file in individual_result_files:
    print(file.opendap_url)

In [ ]:
from pyesgf.search import SearchConnection
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
import time
from tqdm.auto import tqdm
import os
import re
import xarray as xa

os.environ["ESGF_PYCLIENT_NO_FACETS_STAR_WARNING"] = "on"

def search_esgf_datasets(search_params, max_results=None, connection_url='https://esgf-node.llnl.gov/esg-search'):
    """
    Search ESGF for datasets matching the given parameters.
    
    Args:
        search_params (dict): Search parameters
        max_results (int, optional): Maximum number of results to return
        connection_url (str): ESGF node URL
    
    Returns:
        list: List of dataset results
    """
    conn = SearchConnection(connection_url, distrib=True)
    ctx = conn.new_context(**search_params)
    
    print(f'Number of hits: {ctx.hit_count}')
    
    # Get all results or up to max_results
    if max_results is None:
        results = ctx.search()
    else:
        results = ctx.search(limit=max_results)
        
    return results


def process_json_output(json_output):
    """For any dict values which are lists of length 1, convert to a single value."""
    for key, value in json_output.items():
        if isinstance(value, list) and len(value) == 1:
            json_output[key] = value[0]
    return json_output
    
    
def datasets_to_dataframe(dataset_results):
    """
    Convert dataset results to a pandas DataFrame for easier filtering.
    
    Args:
        dataset_results (list): List of dataset results from pyesgf
        
    Returns:
        pd.DataFrame: DataFrame containing dataset metadata
    """
    datasets = []
    
    for result in tqdm(dataset_results, desc="Extracting result metadata"):
        # Extract key metadata
        metadata = {
            'id': result.dataset_id,
            'number_of_files': result.number_of_files,
        }
        
        # Add all JSON metadata
        metadata.update(process_json_output(result.json))
        datasets.append(metadata)
    
    df = pd.DataFrame(datasets)
    
    # Add standardized resolution if nominal_resolution exists
    if 'nominal_resolution' in df.columns:
        df['resolution_degrees'] = df['nominal_resolution'].apply(standardize_resolution_to_degrees)
    
    return df


def standardize_resolution_to_degrees(resolution):
    """
    Convert resolution strings to decimal degrees concisely.
    
    Args:
        resolution: String (e.g., '100 km', '1x1 degree') or list containing resolution
        
    Returns:
        float: Resolution in decimal degrees
    """
    # Handle None, NaN or empty list
    if not resolution or (isinstance(resolution, list) and not resolution):
        return np.nan
    
    # Handle list input
    if isinstance(resolution, list):
        resolution = resolution[0]
    
    # Convert to lowercase string
    res_str = str(resolution).lower()
    
    # Extract numeric value(s)
    numbers = re.findall(r'\d+(?:\.\d+)?', res_str)
    if not numbers:
        return np.nan
    
    # Already in degrees
    if any(unit in res_str for unit in ['degree', 'deg', '°']):
        # For 'axb degree' format, take the average
        if 'x' in res_str and len(numbers) >= 2:
            return (float(numbers[0]) + float(numbers[1])) / 2
        return float(numbers[0])
    
    # Convert km to degrees (Earth circumference ≈ 40,000 km = 360 degrees)
    if 'km' in res_str:
        return float(numbers[0]) / 111.0
    
    # Default assumption: km if no unit specified
    return float(numbers[0]) / 111.0

def filter_datasets(df, filters):
    """
    Filter datasets based on specified criteria.
    
    Args:
        df (pd.DataFrame): DataFrame of datasets
        filters (dict): Dictionary of column names and filter values/functions
        
    Returns:
        pd.DataFrame: Filtered DataFrame
    """
    filtered_df = df.copy()
    
    for column, filter_value in filters.items():
        if column not in filtered_df.columns:
            print(f"Warning: Column '{column}' not found in DataFrame")
            continue
            
        if callable(filter_value):
            # If filter_value is a function, apply it
            filtered_df = filtered_df[filter_value(filtered_df[column])]
        elif isinstance(filter_value, (list, tuple)):
            # If filter_value is a list, check for membership
            filtered_df = filtered_df[filtered_df[column].isin(filter_value)]
        else:
            # Direct equality comparison
            filtered_df = filtered_df[filtered_df[column] == filter_value]
    
    return filtered_df

def get_dataset_files(dataset_result, file_type='opendap', variables=None):
    """
    Get file URLs for a dataset.
    
    Args:
        dataset_result: pyesgf dataset result
        file_type (str): Type of URL to return ('opendap', 'http', 'gridftp')
        variables (list, optional): Filter for specific variables
        
    Returns:
        list: List of file URLs
    """
    file_context = dataset_result.file_context()
    
    # Apply variable filter if specified
    if variables:
        file_context = file_context.constrain(variable=variables)
    
    files = file_context.search()
    
    if file_type == 'opendap':
        return [f.opendap_url for f in files if hasattr(f, 'opendap_url')]
    elif file_type == 'http':
        return [f.download_url for f in files if hasattr(f, 'download_url')]
    elif file_type == 'gridftp':
        return [f.gridftp_url for f in files if hasattr(f, 'gridftp_url')]
    else:
        return [{'opendap': f.opendap_url if hasattr(f, 'opendap_url') else None,
                 'http': f.download_url if hasattr(f, 'download_url') else None,
                 'gridftp': f.gridftp_url if hasattr(f, 'gridftp_url') else None} 
                for f in files]

def get_files_for_filtered_datasets(dataset_results, filtered_indices, file_type='opendap', 
                                   variables=None, max_workers=5):
    """
    Get files for filtered datasets using parallel processing.
    
    Args:
        dataset_results (list): Original dataset results
        filtered_indices (list): Indices of filtered datasets
        file_type (str): Type of URL to return
        variables (list, optional): Filter for specific variables
        max_workers (int): Maximum number of parallel workers
        
    Returns:
        dict: Dictionary mapping dataset IDs to lists of file URLs
    """
    filtered_datasets = [dataset_results[i] for i in filtered_indices]
    total = len(filtered_datasets)
    
    all_files = {}
    
    # Use ThreadPoolExecutor for parallel processing
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Create a function that processes one dataset and returns ID and files
        def process_dataset(dataset):
            try:
                dataset_id = dataset.dataset_id
                files = get_dataset_files(dataset, file_type, variables)
                return dataset_id, files
            except Exception as e:
                print(f"Error processing {dataset.dataset_id}: {e}")
                return dataset.dataset_id, []
        
        # Submit all tasks and show progress
        future_to_dataset = {executor.submit(process_dataset, ds): ds for ds in filtered_datasets}
        
        # Process results as they complete
        for i, future in enumerate(tqdm(future_to_dataset, desc="Fetching file metadata", total=total)):
            try:
                dataset_id, files = future.result()
                all_files[dataset_id] = files
            except Exception as e:
                print(f"Task failed: {e}")
    
    return all_files


# 1. Set search parameters
search_params = {
    'project': 'CMIP6',
    'experiment_id': ['historical'],
    'variable': ['arag'],
    # 'variable': ['arag', 'tos', 'rsdo', 'so', 'no3', 'po4'],
    'frequency': 'mon',
}

# 2. Perform the search
dataset_results = search_esgf_datasets(search_params, max_results=10,
                                    #    connection_url='http://esgf-data.dkrz.de/esg-search' # takes forever, 6 hits
                                       connection_url='https://esgf-node.llnl.gov/esg-search' # faster, 83 hits, no files
                                    #    connection_url='https://esgf.ceda.ac.uk/esg-search'  # slow, no hits, no files
                                    #    connection_url='https://esgf.ceda.ac.uk/esg-search'  # slow, no hits, no files
                                       )

# 3. Convert to DataFrame for filtering
df = datasets_to_dataframe(dataset_results)

# 4. Apply filters (examples)
filters = {
    # Get high-resolution datasets (less than 1 degree)
    'resolution_degrees': lambda x: x < 5.0,
    # Only get specific experiments
    # 'experiment_id': ['ssp126', 'ssp585'],
    # 'variant_label': ['r8i1p1f1']
    # Only get datasets with ocean biogeochemistry
    # 'realm': lambda x: x.apply(lambda y: 'ocnBgchem' in y if isinstance(y, list) else y == 'ocnBgchem')
}

filtered_df = filter_datasets(df, filters)
print(f"Filtered from {len(df)} to {len(filtered_df)} datasets")

# 5. Get files for filtered datasets (only for the first 5 for example)
filtered_indices = filtered_df.index.tolist()
dataset_files = get_files_for_filtered_datasets(
    dataset_results, 
    filtered_indices,
    file_type='opendap',
    variables=search_params['variable']  # Only get files for specific variables
)

# 6. Print results
for dataset_id, files in dataset_files.items():
    print(f"\nDataset: {dataset_id}")
    print(f"Number of files: {len(files)}")
    for file in files[:3]:  # Show first 3 files
        print(f"  {file}")
    if len(files) > 3:
        print(f"  ... ({len(files) - 3} more files)")

Number of hits: 83


Extracting result metadata:   0%|          | 0/83 [00:00<?, ?it/s]

Filtered from 83 to 83 datasets


Fetching file metadata:   0%|          | 0/83 [00:00<?, ?it/s]


Dataset: CMIP6.CMIP.CSIRO.ACCESS-ESM1-5.historical.r3i1p1f1.Omon.arag.gn.v20191203|esgf-data1.llnl.gov
Number of files: 17
  http://esgf-data1.llnl.gov/thredds/dodsC/css03_data/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/historical/r3i1p1f1/Omon/arag/gn/v20191203/arag_Omon_ACCESS-ESM1-5_historical_r3i1p1f1_gn_185001-185912.nc
  http://esgf-data1.llnl.gov/thredds/dodsC/css03_data/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/historical/r3i1p1f1/Omon/arag/gn/v20191203/arag_Omon_ACCESS-ESM1-5_historical_r3i1p1f1_gn_186001-186912.nc
  http://esgf-data1.llnl.gov/thredds/dodsC/css03_data/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/historical/r3i1p1f1/Omon/arag/gn/v20191203/arag_Omon_ACCESS-ESM1-5_historical_r3i1p1f1_gn_187001-187912.nc
  ... (14 more files)

Dataset: CMIP6.CMIP.CSIRO.ACCESS-ESM1-5.historical.r10i1p1f1.Omon.arag.gn.v20200605|esgf-data1.llnl.gov
Number of files: 17
  https://esgf-data1.llnl.gov/thredds/dodsC/css03_data/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/historical/r10i1p1f1/Omon/arag/gn/v20200605/arag_Omon_ACCESS-ESM1-5

In [20]:
filtered_df

,id,number_of_files,version,access,activity_drs,activity_id,cf_standard_name,citation_url,data_node,data_specs_version,...,variant_label,xlink,_version_,retracted,_timestamp,score,creation_date,short_description,datetime_end,resolution_degrees
0,CMIP6.CMIP.CSIRO.ACCESS-ESM1-5.historical.r3i1...,17,20191203,"[HTTPServer, GridFTP, OPENDAP, Globus, LAS]",CMIP,CMIP,mole_concentration_of_aragonite_expressed_as_c...,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf-data1.llnl.gov,01.00.30,...,r3i1p1f1,[http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6...,1661425045908488192,False,2020-03-17T15:19:54.917Z,1.0,NaN,NaN,NaN,2.252252
1,CMIP6.CMIP.CSIRO.ACCESS-ESM1-5.historical.r10i...,17,20200605,"[HTTPServer, OPENDAP, GridFTP, Globus]",CMIP,CMIP,mole_concentration_of_aragonite_expressed_as_c...,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf-data1.llnl.gov,01.00.30,...,r10i1p1f1,[http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6...,1674877993339584512,False,2020-08-13T03:08:45.362Z,1.0,2020-06-05T05:34:04Z,ACCESS-ESM1-5 output prepared for CMIP6,2014-12-16T00:00:00Z,2.252252
2,CMIP6.CMIP.CSIRO.ACCESS-ESM1-5.historical.r9i1...,17,20200529,"[HTTPServer, OPENDAP, GridFTP, Globus]",CMIP,CMIP,mole_concentration_of_aragonite_expressed_as_c...,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf-data1.llnl.gov,01.00.30,...,r9i1p1f1,[http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6...,1674871723946672128,False,2020-08-13T01:29:06.403Z,1.0,2020-05-29T07:33:57Z,ACCESS-ESM1-5 output prepared for CMIP6,2014-12-16T00:00:00Z,2.252252
3,CMIP6.CMIP.CSIRO.ACCESS-ESM1-5.historical.r4i1...,17,20200529,"[HTTPServer, OPENDAP, GridFTP, Globus]",CMIP,CMIP,mole_concentration_of_aragonite_expressed_as_c...,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf-data1.llnl.gov,01.00.30,...,r4i1p1f1,[http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6...,1674871847714291712,False,2020-08-13T01:31:04.437Z,1.0,2020-05-29T07:44:00Z,ACCESS-ESM1-5 output prepared for CMIP6,2014-12-16T00:00:00Z,2.252252
4,CMIP6.CMIP.CSIRO.ACCESS-ESM1-5.historical.r5i1...,17,20200601,"[HTTPServer, OPENDAP, GridFTP, Globus]",CMIP,CMIP,mole_concentration_of_aragonite_expressed_as_c...,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf-data1.llnl.gov,01.00.30,...,r5i1p1f1,[http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6...,1674863085668532224,False,2020-08-12T23:11:48.299Z,1.0,2020-06-01T02:53:19Z,ACCESS-ESM1-5 output prepared for CMIP6,2014-12-16T00:00:00Z,2.252252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78,CMIP6.CMIP.CSIRO.ACCESS-ESM1-5.historical.r16i...,17,20200803,"[HTTPServer, GridFTP, OPENDAP, Globus]",CMIP,CMIP,mole_concentration_of_aragonite_expressed_as_c...,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf.nci.org.au,01.00.30,...,r16i1p1f1,[http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6...,1823922701264224256,False,2020-10-11T14:47:59.389Z,1.0,NaN,NaN,NaN,2.252252
79,CMIP6.CMIP.CSIRO.ACCESS-ESM1-5.historical.r10i...,17,20200605,"[HTTPServer, GridFTP, OPENDAP, Globus]",CMIP,CMIP,mole_concentration_of_aragonite_expressed_as_c...,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf.nci.org.au,01.00.30,...,r10i1p1f1,[http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6...,1823921550603059200,False,2020-06-22T16:49:55.587Z,1.0,NaN,NaN,NaN,2.252252
80,CMIP6.CMIP.CSIRO.ACCESS-ESM1-5.historical.r17i...,17,20200803,"[HTTPServer, GridFTP, OPENDAP, Globus]",CMIP,CMIP,mole_concentration_of_aragonite_expressed_as_c...,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf.nci.org.au,01.00.30,...,r17i1p1f1,[http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6...,1823922568756723712,False,2020-10-11T14:55:34.954Z,1.0,NaN,NaN,NaN,2.252252
81,CMIP6.CMIP.CSIRO.ACCESS-ESM1-5.historical.r14i...,17,20200803,"[HTTPServer, GridFTP, OPENDAP, Globus]",CMIP,CMIP,mole_concentration_of_aragonite_expressed_as_c...,http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6....,esgf.nci.org.au,01.00.30,...,r14i1p1f1,[http://cera-www.dkrz.de/WDCC/meta/CMIP6/CMIP6...,1823923257108070400

In [35]:
dataset_files[list(dataset_files.keys())[1]][:3]

['https://esgf-data1.llnl.gov/thredds/dodsC/css03_data/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/historical/r10i1p1f1/Omon/arag/gn/v20200605/arag_Omon_ACCESS-ESM1-5_historical_r10i1p1f1_gn_185001-185912.nc',
 'https://esgf-data1.llnl.gov/thredds/dodsC/css03_data/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/historical/r10i1p1f1/Omon/arag/gn/v20200605/arag_Omon_ACCESS-ESM1-5_historical_r10i1p1f1_gn_186001-186912.nc',
 'https://esgf-data1.llnl.gov/thredds/dodsC/css03_data/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/historical/r10i1p1f1/Omon/arag/gn/v20200605/arag_Omon_ACCESS-ESM1-5_historical_r10i1p1f1_gn_187001-187912.nc']

In [39]:
ds = xa.open_mfdataset(dataset_files[list(dataset_files.keys())[1]][:], combine='by_coords', engine='netcdf4', chunks={'time':1})

KeyboardInterrupt: 

In [42]:
dataset_files[list(dataset_files.keys())[1]][0]

'https://esgf-data1.llnl.gov/thredds/dodsC/css03_data/CMIP6/CMIP/CSIRO/ACCESS-ESM1-5/historical/r10i1p1f1/Omon/arag/gn/v20200605/arag_Omon_ACCESS-ESM1-5_historical_r10i1p1f1_gn_185001-185912.nc'

In [ ]:
sds = xa.open_dataset(dataset_files[list(dataset_files.keys())[1]][0], engine='netcdf4', chunks={'time':1})

In [30]:
500/20

25.0

In [29]:
ds

<xarray.Dataset> Size: 56GB
Dimensions:             (time: 1980, bnds: 2, lev: 50, j: 300, i: 360,
                         vertices: 4)
Coordinates:
  * time                (time) datetime64[ns] 16kB 1850-01-16T12:00:00 ... 20...
  * lev                 (lev) float64 400B 5.0 15.0 25.0 ... 5.499e+03 5.831e+03
  * j                   (j) int32 1kB 0 1 2 3 4 5 6 ... 294 295 296 297 298 299
  * i                   (i) int32 1kB 0 1 2 3 4 5 6 ... 354 355 356 357 358 359
    latitude            (j, i) float64 864kB dask.array<chunksize=(300, 360), meta=np.ndarray>
    longitude           (j, i) float64 864kB dask.array<chunksize=(300, 360), meta=np.ndarray>
Dimensions without coordinates: bnds, vertices
Data variables:
    time_bnds           (time, bnds) datetime64[ns] 32kB dask.array<chunksize=(1, 2), meta=np.ndarray>
    lev_bnds            (time, lev, bnds) float64 2MB dask.array<chunksize=(120, 50, 2), meta=np.ndarray>
    vertices_latitude   (time, j, i, vertices) float64 7GB dask.array<chunksize=(120, 300, 360, 4), meta=np.ndarray>
    vertices_longitude  (time, j, i, vertices) float64 7GB dask.array<chunksize=(120, 300, 360, 4), meta=np.ndarray>
    arag                (time, lev, j, i) float32 43GB dask.array<chunksize=(1, 50, 300, 360), meta=np.ndarray>
Attributes: (12/48)
    Conventions:                     CF-1.7 CMIP-6.2
    activity_id:                     CMIP
    branch_method:                   standard
    branch_time_in_child:            0.0
    branch_time_in_parent:           87658.0
    creation_date:                   2020-06-05T05:21:24Z
    ...                              ...
    variant_label:                   r10i1p1f1
    version:                         v20200605
    license:                         CMIP6 model data produced by CSIRO is li...
    cmor_version:                    3.4.0
    tracking_id:                     hdl:21.14100/5bb2cbae-3ece-45ef-8b97-79b...
    DODS_EXTRA.Unlimited_Dimension:  time

In [28]:
# average over each month
monthly_mean = ds.groupby('time.month').mean(dim='time')
# average spatially
spatial_mean = monthly_mean.mean(dim=['j', 'i'])
# average over time
mean = monthly_mean.mean(dim='month')
# plot a line for each month, with mean arag value as a function of lev
mean.arag.plot(x='lev', y='arag')

KeyboardInterrupt: 

In [ ]:
xa.open_mfdataset(dataset_files[list(dataset_files.keys())[1][:2]], engine='netcdf4', chunks={'time':1})

In [ ]:
test_fps = ['http://esgf-data3.diasjp.net/thredds/dodsC/esg_dataroot/CMIP6/CMIP/NCAR/CESM2/historical/r10i1p1f1/Amon/tas/gn/v20190313/tas_Amon_CESM2_historical_r10i1p1f1_gn_185001-189912.nc',
 'http://esgf-data3.diasjp.net/thredds/dodsC/esg_dataroot/CMIP6/CMIP/NCAR/CESM2/historical/r10i1p1f1/Amon/tas/gn/v20190313/tas_Amon_CESM2_historical_r10i1p1f1_gn_190001-194912.nc',
 'http://esgf-data3.diasjp.net/thredds/dodsC/esg_dataroot/CMIP6/CMIP/NCAR/CESM2/historical/r10i1p1f1/Amon/tas/gn/v20190313/tas_Amon_CESM2_historical_r10i1p1f1_gn_195001-199912.nc',
 'http://esgf-data3.diasjp.net/thredds/dodsC/esg_dataroot/CMIP6/CMIP/NCAR/CESM2/historical/r10i1p1f1/Amon/tas/gn/v20190313/tas_Amon_CESM2_historical_r10i1p1f1_gn_200001-201412.nc']
ds = xr.open_mfdataset(files_to_open, combine='by_coords')


In [ ]:
# save ds as local netcdf file
ds.to_netcdf('test.nc')

In [ ]:
# average over each month
monthly_mean = ds.groupby('time.month').mean(dim='time')
monthly_mean

In [ ]:
import xesmf as xe
import numpy as np

def regrid_to_regular(ds, variable='so', lat_res=1.0, lon_res=1.0):
    """
    Regrid a CMIP6 dataset with (i, j) coordinates to a regular lat/lon grid using xesmf.
    
    Parameters:
        ds (xr.Dataset): Input dataset with irregular grid.
        variable (str): Name of the variable to regrid.
        lat_res (float): Latitude resolution in degrees.
        lon_res (float): Longitude resolution in degrees.
    
    Returns:
        xr.Dataset: Dataset on a regular grid.
    """

    # Extract original lat/lon
    lat = ds['latitude']
    lon = ds['longitude']
    
    # Construct source grid (must be 2D)
    source_grid = {'lon': lon.values, 'lat': lat.values}

    # Define regular grid (target)
    lat_out = np.arange(-89.5, 90, lat_res)
    lon_out = np.arange(0.5, 360, lon_res)  # CMIP6 often uses 0–360
    target_grid = xr.Dataset({
        'lat': (['lat'], lat_out),
        'lon': (['lon'], lon_out),
    })

    # Build regridder (conservative can also be used if conserving mass is important)
    regridder = xe.Regridder(ds, target_grid, method='bilinear', periodic=True, reuse_weights=True)

    # Regrid the variable
    da_regridded = regridder(ds[variable])

    return da_regridded



In [ ]:
dataset_files[list(dataset_files.keys())[1]]

In [ ]:
dataset_files['CMIP6.ScenarioMIP.DKRZ.MPI-ESM1-2-HR.ssp370.r8i1p1f1.Omon.po4.gn.v20190710|esgf-data1.llnl.gov']

In [ ]:
df.variant_label

In [ ]:
df.numerical_resolution_degrees

In [ ]:
xa.open_dataset("http://esgf-data1.llnl.gov/thredds/dodsC/css03_data/CMIP6/ScenarioMIP/DKRZ/MPI-ESM1-2-HR/ssp370/r8i1p1f1/Omon/po4/gn/v20190710/po4_Omon_MPI-ESM1-2-HR_ssp370_r8i1p1f1_gn_201501-201912.nc")

In [ ]:
def standardize_esgf_metadata(dict_list):
    """
    Standardize a list of ESGF metadata dictionaries for easy DataFrame creation.
    
    Args:
        dict_list (list): List of dictionaries containing ESGF metadata
        
    Returns:
        pd.DataFrame: Standardized DataFrame with consistent columns
    """
    if not dict_list:
        return pd.DataFrame()
    
    # Create a default dictionary to track all possible keys
    all_keys = set()
    for d in tqdm(dict_list, desc="Collecting keys"):
        all_keys.update(d.json.keys())
    
    # Add keys for access types
    for access_type in ['HTTPServer', 'GridFTP', 'OPENDAP', 'Globus']:
        all_keys.add(access_type.lower() + '_access')
    
    # Create a standardized list of dictionaries with consistent keys
    standardized_dicts = []
    
    for d in tqdm(dict_list, desc="Standardizing dictionaries"):
        # Create a new dictionary with all possible keys
        standardized = {key: None for key in all_keys}
        
        # Update with values from the original dict
        for key, value in d.json.items():
            if key == 'access' and isinstance(value, list):
                for access_type in ['HTTPServer', 'OPENDAP', 'GridFTP', 'Globus']:
                    standardized['access_' + access_type.lower()] = access_type in value
            elif isinstance(value, list) and len(value) == 1:
                standardized[key] = value[0]
            else:
                standardized[key] = value
        
        standardized_dicts.append(standardized)
    
    # Convert to DataFrame
    df = pd.DataFrame(standardized_dicts)
    
    # Handle list columns - optionally convert single-item lists to scalars
    for col in df.columns:
        if df[col].apply(lambda x: isinstance(x, list) and len(x) == 1).any():
            df[col] = df[col].apply(lambda x: x[0] if isinstance(x, list) and len(x) == 1 else x)
    
    return df


test_df = standardize_esgf_metadata(result)
# drop duplicates
test_df.drop_duplicates(subset=['master_id'], inplace=True)
# standardise the resolution column
test_df['numerical_nominal_resolution'] = test_df['nominal_resolution'].apply(standardize_resolution_to_degrees)

In [ ]:
result[0].

In [ ]:
test_df.columns

In [ ]:
test_df[['access_opendap', 'access_httpserver', 'access_gridftp', 'access_globus']].access_httpserver.value_counts()

In [ ]:
test_df.columns

In [ ]:
xa.open_dataset(test_df[test_df.data_node == "aims3.llnl.gov"].url.iloc[0], engine='netcdf4')

In [ ]:
test_df.iloc[0]

In [ ]:
test_df.data_node.value_counts()

In [ ]:
test_df.url[100]

In [ ]:
xa.open_dataset("https://app.globus.org/file_manager?origin_id=1889ea03-25ad-4f9f-8110-1ce8833a9d7e&origin_path=/css03_data/CMIP6/CMIP/MOHC/HadGEM3-GC31-LL/historical/r18i1p1f3/Omon/so/gn/v20230515")
# xa.open_dataset(test_df.url.iloc[0][1])

In [ ]:
test_df.columns

In [ ]:
test_df.access

In [ ]:
test_df.url.iloc[0]

# Deprecated

In [ ]:
conn = SearchConnection('https://esgf.ceda.ac.uk/esg-search',
                        distrib=True)   # distrib=True searches all ESGF nodes, not just CEDA

facets = 'project,institution_id,experiment_id,variant_label,variable_id'
ctx = conn.new_context(
    facets=facets,
    project='CMIP6',
    institution_id='AWI',
    experiment_id='historical,ssp585',
    variant_label='r1i1p1f1',
    variable_id='tos',
    table_id='Omon',
    nominal_resolution='25 km'
    )

results = ctx.search()
print(f"Found {len(results)} results")

In [ ]:



### create dataframe of results for visualisation and checking for duplicates
files = []
# for i in tqdm(range(2)):
for i in tqdm(range(len(results))):
    try:
        hit = results[i].file_context().search()
    except:
        hit = results[i].file_context().search()
        
    for f in hit:
        file_dict = {
            'filename': f.filename,
            'download_url': f.download_url,
            'opendap_url': f.opendap_url
        }
        file_dict.update(process_hit_json(f.json))
        files.append(file_dict)



In [ ]:
df_info = pd.DataFrame(files)
df_info.drop_duplicates(subset=['filename'], inplace=True)

In [ ]:
df_info.columns

In [ ]:
# df_info.filename.unique()
df_info.further_info_url.unique()

In [ ]:
file_list = df_info.opendap_url.unique()
file_list

In [ ]:
file_list[2:4]

In [ ]:
xa.open_dataset(file_list[1])

In [ ]:
xa.open_mfdataset(file_list[2:4], chunks={'time': 120})

In [ ]:
dl_info.drop_duplicates(subset=['filename']).filename.value_counts()

In [ ]:
new_conn = SearchConnection('https://esgf-data.dkrz.de/esg-search', distrib=True)

ctx = new_conn.new_context(
    project='CMIP6',
    source_id='UKESM1-0-LL',
    experiment_id='historical',
    variable='tas',
    frequency='mon',
    variant_label='r1i1p1f1',
    # data_node='esgf-data3.ceda.ac.uk'
    )
ctx.hit_count

In [ ]:
result = ctx.search()[0]
result.dataset_id

In [ ]:
files = result.file_context().search()
print(len(files))

In [ ]:
files[0].opendap_url

In [ ]:
xa.open_dataset(files[0].opendap_url, chunks={'time': 120})